In [ ]:
# ============================================================
# SEM Dashboard (Offline HTML) — FINAL
# - model_coefficients.csv: regression paths (~) and covariances (~~)
# - factor_loadings.csv: factor loadings (λ = Est. Std) + p + residual variance (optional)
# - model_fit_statistics.csv: CFI / GFI / RMSEA (optional)
# Output: sem_dashboard.html
# ============================================================

import pandas as pd
import numpy as np
import json, math, textwrap
from math import sqrt
import plotly.graph_objects as go
from plotly.offline import get_plotlyjs

# =========================
# INPUTS (alleen dit aanpassen)
# =========================
COEF_CSV_PATH     = "model_coefficients.csv"
LOADINGS_CSV_PATH = "factor_loadings.csv"
FIT_CSV_PATH      = "model_fit_statistics.csv"
OUTPUT_HTML       = "sem_dashboard.html"

# =========================
# HELPERS
# =========================
def safe_float(x):
    if x is None:
        return None
    s = str(x).strip()
    if s in ["", "-", "NA", "NaN", "nan", "None", "null"]:
        return None
    try:
        return float(s)
    except:
        return None

def sanitize(obj):
    """Make plotly dict JSON safe: convert NaN/inf to None."""
    if isinstance(obj, dict): return {k: sanitize(v) for k, v in obj.items()}
    if isinstance(obj, list): return [sanitize(v) for v in obj]
    if isinstance(obj, tuple): return [sanitize(v) for v in obj]
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj): return None
        return obj
    return obj

def norm_colnames(df):
    return {c.lower().strip(): c for c in df.columns}

def req_col(df, name):
    cols = norm_colnames(df)
    key = name.lower().strip()
    if key not in cols:
        raise ValueError(f"Kolom '{name}' ontbreekt. Gevonden: {list(df.columns)}")
    return cols[key]

def find_col(cols_map, possibles):
    for p in possibles:
        k = p.lower().strip()
        if k in cols_map:
            return cols_map[k]
    return None

def wrap_html(text, width=14, max_lines=3):
    t = str(text).replace("_", " ").strip()
    lines = textwrap.wrap(t, width=width)[:max_lines]
    return "<br>".join(lines) if lines else t

def wrap_indicator(text, width=14, max_lines=2):
    t = str(text).replace("_", " ").strip()
    lines = textwrap.wrap(t, width=width)[:max_lines]
    return "<br>".join(lines) if lines else t

def p_to_stars(p):
    p = safe_float(p)
    if p is None: return ""
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return ""

def clamp(a, lo, hi):
    return max(lo, min(hi, a))

def edge_width(beta):
    b = abs(beta) if beta is not None else 0.0
    return 1.4 + 2.6 * clamp(b, 0.0, 0.9)

def dist(a, b):
    return sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

def unit_normal(dx, dy):
    L = sqrt(dx*dx + dy*dy) if (dx*dx + dy*dy) else 1.0
    return (-dy/L, dx/L)

def edge_points(p0, p1, r):
    """Return start/end points on circle edges (not center-to-center)."""
    x0, y0 = p0
    x1, y1 = p1
    dx, dy = x1 - x0, y1 - y0
    L = sqrt(dx*dx + dy*dy)
    if L == 0:
        return p0, p1
    ux, uy = dx / L, dy / L
    start = (x0 + ux * r, y0 + uy * r)
    end   = (x1 - ux * r, y1 - uy * r)
    return start, end

# =========================
# READ: model_coefficients.csv
# =========================
coef_df = pd.read_csv(COEF_CSV_PATH)
c_lval = req_col(coef_df, "lval")
c_op   = req_col(coef_df, "op")
c_rval = req_col(coef_df, "rval")

cols_coef = norm_colnames(coef_df)
c_est = find_col(cols_coef, ["estimate", "Estimate"])
if c_est is None:
    raise ValueError("Kolom 'Estimate' ontbreekt in model_coefficients.csv")

c_p = find_col(cols_coef, ["p", "p-value", "pvalue", "p_value"])

keep = [c_lval, c_op, c_rval, c_est] + ([c_p] if c_p else [])
work = coef_df[keep].copy()
work.columns = ["lval","op","rval","estimate"] + (["p"] if c_p else [])
work["estimate"] = work["estimate"].apply(safe_float)
if "p" in work.columns:
    work["p"] = work["p"].apply(safe_float)

reg = work[work["op"] == "~"].copy()
var = work[work["op"] == "~~"].copy()

# =========================
# AUTO DETECT LATENTS
# =========================
predictors = set(reg["rval"].astype(str))
outcomes   = set(reg["lval"].astype(str))
variances  = set(var.loc[var["lval"].astype(str) == var["rval"].astype(str), "lval"].astype(str))

latents = sorted([x for x in predictors if (x in outcomes) or (x in variances)])
if len(latents) == 0:
    latents = list(reg["rval"].value_counts().head(6).index.astype(str))
latent_set = set(latents)

# JD-R friendly names
LABELS = {
    "Stressors": "Stressoren",
    "Energy_Sources": "Energiebronnen",
    "Response_to_Stress": "Stressreacties",
    "Wellbeing": "Welbevinden",
    "Negative_Outcomes": "Negatieve uitkomsten",
    "Positive_Outcomes": "Positieve uitkomsten",
}
def latent_title(x):
    return LABELS.get(x, x.replace("_"," "))

# =========================
# SPLIT PATHS + COVARIANCES
# =========================
# regressiepaden tussen latents
paths = reg[(reg["lval"].astype(str).isin(latent_set)) & (reg["rval"].astype(str).isin(latent_set))].copy()
paths = paths[paths["estimate"].notna()]

# covarianties tussen latents (A ~~ B, A!=B)
cov_lat = var[
    (var["lval"].astype(str).isin(latent_set)) &
    (var["rval"].astype(str).isin(latent_set)) &
    (var["lval"].astype(str) != var["rval"].astype(str))
].copy()

cov_map = {}
for _, rr in cov_lat.iterrows():
    a = str(rr["lval"]); b = str(rr["rval"])
    est = safe_float(rr["estimate"])
    pv  = safe_float(rr["p"]) if "p" in cov_lat.columns else None
    if est is None:
        continue
    cov_map[(a,b)] = (est, pv)
    cov_map[(b,a)] = (est, pv)

# Jij wil alleen deze 3 covarianties tekenen (uit jouw screenshot)
COV_PAIRS = [
    ("Response_to_Stress", "Wellbeing"),
    ("Stressors", "Energy_Sources"),
    ("Negative_Outcomes", "Positive_Outcomes"),
]

# =========================
# READ: factor_loadings.csv (λ = Est. Std)
# =========================
load_df = pd.read_csv(LOADINGS_CSV_PATH)
cols_load = norm_colnames(load_df)

LVAL_COL   = find_col(cols_load, ["lval", "indicator", "item", "observed"])
RVAL_COL   = find_col(cols_load, ["rval", "latent", "factor"])
ESTSTD_COL = find_col(cols_load, ["est. std", "est.std", "std.all", "standardized", "std_estimate"])
EST_COL    = find_col(cols_load, ["estimate", "est"])
P_COL      = find_col(cols_load, ["p", "p-value", "pvalue", "p_value"])
RESID_COL  = find_col(cols_load, ["resid", "residual", "residual variance", "residual_variance"])

if LVAL_COL is None or RVAL_COL is None:
    raise ValueError("factor_loadings.csv mist indicator/latent kolommen (lval/rval).")

# We gebruiken Est. Std als loading (zoals jij vroeg)
if ESTSTD_COL is None:
    if EST_COL is None:
        raise ValueError("factor_loadings.csv mist zowel 'Est. Std' als 'Estimate'.")
    ESTSTD_COL = EST_COL  # fallback, maar liever Est. Std

sel_cols = [LVAL_COL, RVAL_COL, ESTSTD_COL] + ([P_COL] if P_COL else []) + ([RESID_COL] if RESID_COL else [])
load_work = load_df[sel_cols].copy()
load_work.columns = ["lval","rval","loading"] + (["p"] if P_COL else []) + (["resid"] if RESID_COL else [])

load_work["loading"] = load_work["loading"].apply(safe_float)
if "p" in load_work.columns:
    load_work["p"] = load_work["p"].apply(safe_float)

if "resid" in load_work.columns:
    load_work["resid"] = load_work["resid"].apply(safe_float)
else:
    load_work["resid"] = np.nan

# Als resid ontbreekt: resid ≈ 1 - λ^2 (bij gestandaardiseerde oplossing)
calc_resid = load_work["loading"].apply(lambda x: np.nan if x is None else max(0.0, 1.0 - float(x)**2))
load_work["resid"] = load_work["resid"].where(load_work["resid"].notna(), calc_resid)

# Filter alleen latents die we tonen
loadings = load_work[load_work["rval"].astype(str).isin(latent_set)].copy()
loadings = loadings[loadings["loading"].notna()]

# =========================
# READ: model_fit_statistics.csv (CFI / GFI / RMSEA)
# =========================
fit_map = {}
try:
    fit_df = pd.read_csv(FIT_CSV_PATH)
    cols_fit = norm_colnames(fit_df)
    metric_col = find_col(cols_fit, ["metric", "name", "fit", "index"])
    value_col  = find_col(cols_fit, ["value", "estimate", "val"])
    if metric_col and value_col:
        tmp = fit_df[[metric_col, value_col]].copy()
        tmp.columns = ["metric","value"]
        for _, r in tmp.iterrows():
            m = str(r["metric"]).strip().upper()
            v = safe_float(r["value"])
            fit_map[m] = v
    else:
        for k in ["CFI","GFI","RMSEA"]:
            ck = k.lower()
            if ck in cols_fit:
                fit_map[k] = safe_float(fit_df[cols_fit[ck]].iloc[0])
except Exception:
    fit_map = {}

def fmt_fit(name):
    v = fit_map.get(name)
    return "—" if v is None else f"{v:.3f}"

# =========================
# POSITIES (vaste nette JD-R layout)
# =========================
preferred_pos = {
    "Stressors": (-9.0,  3.6),
    "Response_to_Stress": (0.0,  3.6),
    "Negative_Outcomes": ( 9.0,  3.6),
    "Energy_Sources": (-9.0, -3.6),
    "Wellbeing": (0.0, -3.6),
    "Positive_Outcomes": ( 9.0, -3.6),
}
pos = {}
unknown = []
for l in latents:
    if l in preferred_pos:
        pos[l] = preferred_pos[l]
    else:
        unknown.append(l)

# Onbekenden rechts parkeren
if unknown:
    cx, cy = 14.0, 0.0
    R = 4.2
    for i, l in enumerate(unknown):
        ang = 2*np.pi*i/max(1, len(unknown))
        pos[l] = (cx + R*np.cos(ang), cy + R*np.sin(ang))

node_centers = {l: pos[l] for l in latents}

# =========================
# LABEL PLACEMENT (logisch: boven/bovenste rij, onder/onderste rij, cov aan zijkant)
# =========================
NODE_EDGE_R = 1.95      # randafstand voor pijlen/cov
LABEL_PAD   = 0.85      # hoe ver label van lijn af staat
CLEARANCE   = 0.70      # push labels weg van bollen
LABEL_BOX   = 26       # marker size (label box)

def push_away_from_nodes(x, y):
    for c in node_centers.values():
        d = dist((x,y), c)
        min_d = NODE_EDGE_R + CLEARANCE
        if d < min_d:
            vx, vy = x - c[0], y - c[1]
            L = sqrt(vx*vx + vy*vy) if (vx*vx + vy*vy) else 1.0
            push = (min_d - d) + 0.20
            x += (vx/L) * push
            y += (vy/L) * push
    return x, y

def label_pos_for_path(src_center, dst_center, idx):
    x0, y0 = src_center
    x1, y1 = dst_center
    dx, dy = x1 - x0, y1 - y0
    nx, ny = unit_normal(dx, dy)

    # horizontaal vs diagonaal
    is_horizontal = abs(dy) < 0.25 and abs(dx) > 2.0

    same_row_top = (y0 >= 0 and y1 >= 0)
    same_row_bot = (y0 <= 0 and y1 <= 0)

    # 1) positie langs de pijl
    if is_horizontal:
        t = 0.50
    else:
        t = 0.30  # stabiel: dichter bij bron dan midden

    mx = x0 + dx * t
    my = y0 + dy * t

    # 2) offset loodrecht op de lijn
    # top labels boven de lijn, bottom labels onder de lijn
    sign = 1 if same_row_top else -1

    # top iets dichter, bottom iets verder (zodat het niet in de bol komt)
    pad = LABEL_PAD * (0.60 if same_row_top else 0.88)

    # kleine spreiding zodat labels niet exact op elkaar staan
    stagger = (idx % 3 - 1) * 0.16

    lx = mx + nx * (pad + stagger) * sign
    ly = my + ny * (pad + stagger) * sign

    # 3) duw weg van bollen (gebruik jouw bestaande functie)
    lx, ly = push_away_from_nodes(lx, ly)

    return lx, ly



def label_pos_for_cov(a, b):
    """Cov labels: zet ze aan de zijkant i.p.v. midden tussen kruisingen."""
    x0,y0 = a
    x1,y1 = b
    dx,dy = x1-x0, y1-y0
    nx,ny = unit_normal(dx,dy)
    mx,my = (x0+x1)/2, (y0+y1)/2

    # voor verticale covs (zoals links en rechts): label naar links/rechts
    if abs(dx) < 0.7:
        # links kolom -> label links, rechts kolom -> label rechts
        sign = -1 if mx < 0 else 1
        lx = mx + sign * 1.55
        ly = my + 0.00
    else:
        # andere: gewoon normaal
        lx = mx + nx * 1.35
        ly = my + ny * 1.35

   
    return lx, ly

# =========================
# OVERVIEW FIG (paden + covs + labels + bollen)
# =========================
RED_LINE = "rgba(220,38,38,0.78)"   # rood
RED_COV  = "rgba(220,38,38,0.55)"   # rood (subtieler)
BG_COL   = "rgba(248,250,255,1)"

edge_traces = []
arrow_ann   = []

path_lab_x, path_lab_y, path_lab_t, path_lab_h = [], [], [], []

# draw regression paths (~)
for idx, r in paths.reset_index(drop=True).iterrows():
    s = str(r["rval"]); t = str(r["lval"])
    beta = safe_float(r["estimate"])
    pval = safe_float(r["p"]) if "p" in paths.columns else None

    if beta is None or s not in pos or t not in pos:
        continue

    src_c = pos[s]; dst_c = pos[t]
    start_pt, end_pt = edge_points(src_c, dst_c, NODE_EDGE_R)

    # line for hover
    edge_traces.append(go.Scatter(
        x=[start_pt[0], end_pt[0]],
        y=[start_pt[1], end_pt[1]],
        mode="lines",
        line=dict(width=edge_width(beta), color=RED_LINE),
        hoverinfo="text",
        hovertext=(
            f"<b>Regressiepad</b><br>{latent_title(s)} → {latent_title(t)}"
            f"<br>β = {beta:+.3f}{p_to_stars(pval)}"
            + (f"<br>p = {pval:.3g}" if pval is not None else "")
        ),
        showlegend=False
    ))

    # arrow head as annotation (end on edge)
    arrow_ann.append(dict(
        x=end_pt[0], y=end_pt[1], ax=start_pt[0], ay=start_pt[1],
        xref="x", yref="y", axref="x", ayref="y",
        showarrow=True,
        arrowhead=3,
        arrowsize=1.05,
        arrowwidth=edge_width(beta),
        arrowcolor=RED_LINE,
        opacity=0.95
    ))

    # label box position (logical: above top row, below bottom row)
    lx, ly = label_pos_for_path(src_c, dst_c, idx)
    path_lab_x.append(lx)
    path_lab_y.append(ly)
    path_lab_t.append(f"{beta:+.2f}{p_to_stars(pval)}")
    path_lab_h.append(
        f"<b>Regressiepad</b><br>{latent_title(s)} → {latent_title(t)}"
        f"<br>β = {beta:+.3f}{p_to_stars(pval)}"
        + (f"<br>p = {pval:.3g}" if pval is not None else "")
    )

path_labels = go.Scatter(
    x=path_lab_x, y=path_lab_y,
    mode="markers+text",
    text=path_lab_t,
    textposition="middle center",
    textfont=dict(size=6.4, color="#111"),
    marker=dict(
        size=LABEL_BOX,
        symbol="square",
        color="rgba(255,255,255,0.98)",
        line=dict(width=1, color="rgba(0,0,0,0.22)")
    ),
    hoverinfo="text",
    hovertext=path_lab_h,
    showlegend=False
)

# draw covariances (only 3 pairs) as dashed red + two arrow heads + value label
cov_traces = []
cov_ann    = []
cov_lab_x, cov_lab_y, cov_lab_t, cov_lab_h = [], [], [], []

for (a, b) in COV_PAIRS:
    if a not in pos or b not in pos:
        continue
    ac = pos[a]; bc = pos[b]
    start_pt, end_pt = edge_points(ac, bc, NODE_EDGE_R)

    est, pv = cov_map.get((a,b), (None, None))

    cov_traces.append(go.Scatter(
        x=[start_pt[0], end_pt[0]],
        y=[start_pt[1], end_pt[1]],
        mode="lines",
        line=dict(width=2.2, color=RED_COV, dash="dash"),
        hoverinfo="text",
        hovertext=(
            f"<b>Covariantie</b><br>{latent_title(a)} ↔ {latent_title(b)}"
            + (f"<br>ψ = {est:+.3f}{p_to_stars(pv)}" if est is not None else "<br>ψ = —")
            + (f"<br>p = {pv:.3g}" if pv is not None else "")
        ),
        showlegend=False
    ))

    # two arrow heads (both directions) on edges
    cov_ann.append(dict(
        x=end_pt[0], y=end_pt[1], ax=start_pt[0], ay=start_pt[1],
        xref="x", yref="y", axref="x", ayref="y",
        showarrow=True, arrowhead=2, arrowsize=1.0, arrowwidth=2.2,
        arrowcolor=RED_COV, opacity=0.95
    ))
    cov_ann.append(dict(
        x=start_pt[0], y=start_pt[1], ax=end_pt[0], ay=end_pt[1],
        xref="x", yref="y", axref="x", ayref="y",
        showarrow=True, arrowhead=2, arrowsize=1.0, arrowwidth=2.2,
        arrowcolor=RED_COV, opacity=0.95
    ))

    # label at side
    lx, ly = label_pos_for_cov(ac, bc)
    if est is None:
        txt = "—"
    else:
        txt = f"{est:+.2f}{p_to_stars(pv)}"

    cov_lab_x.append(lx)
    cov_lab_y.append(ly)
    cov_lab_t.append(txt)
    cov_lab_h.append(
        f"<b>Covariantie</b><br>{latent_title(a)} ↔ {latent_title(b)}"
        + (f"<br>ψ = {est:+.3f}{p_to_stars(pv)}" if est is not None else "<br>ψ = —")
        + (f"<br>p = {pv:.3g}" if pv is not None else "")
    )

cov_labels = go.Scatter(
    x=cov_lab_x, y=cov_lab_y,
    mode="markers+text",
    text=cov_lab_t,
    textposition="middle center",
    textfont=dict(size=6.2, color="#111"),
    marker=dict(
        size=LABEL_BOX,
        symbol="square",
        color="rgba(255,255,255,0.98)",
        line=dict(width=1, color="rgba(0,0,0,0.18)")
    ),
    hoverinfo="text",
    hovertext=cov_lab_h,
    showlegend=False
)

# latent nodes (bollen)
node_trace = go.Scatter(
    x=[pos[l][0] for l in latents],
    y=[pos[l][1] for l in latents],
    mode="markers+text",
    text=[wrap_html(latent_title(l), width=14, max_lines=3) for l in latents],
    textposition="middle center",
    textfont=dict(size=13, color="white"),
    marker=dict(size=160, color="#234a6f", line=dict(width=2, color="#0b0c10")),
    hoverinfo="text",
    hovertext=[f"<b>{latent_title(l)}</b><br>Latente variabele (construct): {l}" for l in latents],
    showlegend=False
)

overview_fig = go.Figure(
    data=edge_traces + cov_traces + [path_labels, node_trace]
)
overview_fig.update_layout(
    title="SEM Dashboard – Overzicht",
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    height=760,
    margin=dict(l=20, r=20, t=70, b=20),
    plot_bgcolor=BG_COL,
    paper_bgcolor=BG_COL,
    annotations=arrow_ann + cov_ann
)

overview_node_trace_index = len(overview_fig.data) - 1  # last trace is nodes

# =========================
# DETAIL FIGS (loadings + resid)
# =========================
detail_figs = {}
detail_meta = {}

for latent in latents:
    items = loadings[loadings["rval"].astype(str) == latent].copy()
    items = items.sort_values("loading", ascending=False)

    ind_list = []
    for _, rr in items.iterrows():
        ind = str(rr["lval"])
        ld  = safe_float(rr["loading"])
        pv  = safe_float(rr["p"]) if "p" in items.columns else None
        rv  = safe_float(rr["resid"])
        if ld is None:
            continue
        ind_list.append((ind, ld, pv, rv))

    n = len(ind_list)
    top_n = int(np.ceil(n/2))
    top = ind_list[:top_n]
    bot = ind_list[top_n:]

    spacing_x = 3.35
    top_y = 3.7
    bot_y = -3.7
    cx, cy = 0.0, 0.0

    def place_row(row_items, y):
        nodes = []
        for j, (name, ld, pv, rv) in enumerate(row_items):
            x = (j - (len(row_items)-1)/2) * spacing_x
            nodes.append({"name": name, "x": x, "y": y, "loading": ld, "p": pv, "resid": rv})
        return nodes

    ind_nodes = place_row(top, top_y) + place_row(bot, bot_y)

    # lines from latent center to indicators
    ex, ey = [], []
    for it in ind_nodes:
        ex += [cx, it["x"], None]
        ey += [cy, it["y"], None]

    edges = go.Scatter(
        x=ex, y=ey, mode="lines",
        line=dict(width=1.25, color="rgba(0,0,0,0.45)"),
        hoverinfo="none", showlegend=False
    )

    center = go.Scatter(
        x=[cx], y=[cy], mode="markers+text",
        text=[wrap_html(latent_title(latent), width=14, max_lines=3)],
        textposition="middle center",
        textfont=dict(size=14, color="white"),
        marker=dict(size=170, color="#234a6f", line=dict(width=2, color="#0b0c10")),
        hoverinfo="text",
        hovertext=[f"<b>{latent_title(latent)}</b><br>Latente variabele (construct): {latent}"],
        showlegend=False
    )

    ind_text = [wrap_indicator(it["name"], width=14, max_lines=2) for it in ind_nodes]
    ind_hover = []
    for it in ind_nodes:
        rv = it["resid"]
        rv_txt = "—" if rv is None or (isinstance(rv, float) and np.isnan(rv)) else f"{rv:.3f}"
        ind_hover.append(
            f"<b>{it['name']}</b>"
            f"<br>Factor loading (λ, Est. Std): {it['loading']:+.3f}{p_to_stars(it['p'])}"
            + (f"<br>p = {it['p']:.3g}" if it["p"] is not None else "")
            + f"<br>Residual variance: {rv_txt}"
        )

    inds = go.Scatter(
        x=[it["x"] for it in ind_nodes],
        y=[it["y"] for it in ind_nodes],
        mode="markers+text",
        text=ind_text,
        textposition="middle center",
        textfont=dict(size=10.4, color="#111"),
        marker=dict(size=84, symbol="square", color="#f7e7a0", line=dict(width=1.5, color="#c29c00")),
        hoverinfo="text",
        hovertext=ind_hover,
        showlegend=False
    )

    fig = go.Figure(data=[edges, center, inds])
    fig.update_layout(
        title=f"Detail – {latent_title(latent)}",
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        height=760,
        margin=dict(l=20, r=20, t=70, b=20),
        plot_bgcolor=BG_COL,
        paper_bgcolor=BG_COL,
    )

    detail_figs[latent] = fig
    detail_meta[latent] = {
        "indicatorIndexByName": {it["name"].lower(): i for i, it in enumerate(ind_nodes)},
        "indicatorTraceIndex": 2
    }

# =========================
# BUNDLE FOR HTML
# =========================
bundle = {
    "overviewFig": sanitize(overview_fig.to_dict()),
    "detailFigs": {k: sanitize(v.to_dict()) for k, v in detail_figs.items()},
    "latents": latents,
    "latentTitles": {l: latent_title(l) for l in latents},
    "overviewNodeTraceIndex": overview_node_trace_index,
    "detailMeta": detail_meta,
    "fit": {k: fit_map.get(k) for k in ["CFI","GFI","RMSEA"]},
    "covPairs": COV_PAIRS
}
bundle_json = json.dumps(bundle)
plotly_js = get_plotlyjs()

fit_block_html = f"""
<div class="hr"></div>
<h4 style="margin:8px 0 6px 0; font-size:13px;">Model fit (samenvatting)</h4>
<ul style="margin:6px 0 0 0;">
  <li><b>CFI</b>: {fmt_fit("CFI")}</li>
  <li><b>GFI</b>: {fmt_fit("GFI")}</li>
  <li><b>RMSEA</b>: {fmt_fit("RMSEA")}</li>
</ul>
"""

# =========================
# HTML (offline, simpel)
# =========================
HTML = r"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8"/>
<title>SEM Dashboard</title>
<script>__PLOTLY_JS__</script>
<style>
  :root {
    --bg:#f4f7ff; --card:#fff; --text:#111827; --muted:#6b7280;
    --stroke:rgba(0,0,0,0.10); --shadow:0 14px 38px rgba(0,0,0,0.10);
    --btnbg:rgba(255,255,255,0.92); --btnbd:rgba(0,0,0,0.14);
  }
  body { margin:0; font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif; background:var(--bg); color:var(--text); }
  .wrap { max-width:1400px; margin:18px auto; padding:12px 18px 26px 18px; }
  .top { display:flex; justify-content:space-between; align-items:center; gap:12px; margin-bottom:12px; }
  .h1 { font-size:22px; font-weight:760; }
  .sub { font-size:13px; color:var(--muted); }
  .actions { display:flex; gap:8px; flex-wrap:wrap; align-items:center; }
  .btn { border-radius:999px; border:1px solid var(--btnbd); padding:8px 12px; background:var(--btnbg); cursor:pointer; font-size:13px; user-select:none; }
  .grid { display:grid; grid-template-columns:minmax(0,2.4fr) minmax(420px,1fr); gap:14px; align-items:start; }
  .card { background:var(--card); border-radius:18px; box-shadow:var(--shadow); border:1px solid var(--stroke); padding:12px 14px; }
  #plot { height:760px; }
  .controlrow { display:flex; gap:10px; flex-wrap:wrap; align-items:center; margin:10px 0 6px 0; }
  .chip { border:1px solid var(--stroke); border-radius:999px; padding:6px 10px; font-size:12.5px; color:var(--muted); background: rgba(255,255,255,0.35); }
  .input { flex:1; min-width:240px; border:1px solid var(--stroke); border-radius:12px; padding:10px 12px; background: rgba(255,255,255,0.75); outline:none; color:var(--text); }
  .panel h3 { margin:6px 0 8px 0; font-size:16px; }
  .panel p, .panel li { font-size:12.6px; color:var(--muted); line-height:1.6; }
  .panel ul { margin:8px 0 0 0; padding-left:18px; }
  .hr { height:1px; background:var(--stroke); margin:12px 0; }
</style>
</head>
<body>
<div class="wrap">
  <div class="top">
    <div>
      <div class="h1">SEM Dashboard</div>
      <div class="sub">Overzicht → klik bol → detail → terug • offline HTML</div>
    </div>
    <div class="actions">
      <div class="btn" id="backBtn" style="display:none;">← Terug</div>
      <div class="btn" id="resetBtn">Reset zoom</div>
    </div>
  </div>

  <div class="grid">
    <div class="card">
      <div class="controlrow">
        <input class="input" id="searchBox" placeholder="Zoek: latent (overzicht) of indicator (detail) en druk Enter" />
        <span class="chip" id="crumb">Overzicht</span>
      </div>
      <div id="plot"></div>
    </div>

    <div class="card panel">
      <h3 id="panelTitle">Uitleg (SEM)</h3>
      <div id="panelBody">
        <p>
          In <b>Structural Equation Modeling (SEM)</b> werk je met <b>latente variabelen</b>:
          onderliggende concepten die je niet direct observeert (bijv. stress), maar afleidt uit meerdere
          <b>indicatoren</b> (items/vragen). SEM combineert:
          (1) het <b>measurement model</b> (items → construct) en
          (2) het <b>structural model</b> (constructen → constructen).
        </p>
        <div class="hr"></div>
        <ul>
          <li><b>Regressiepad (→)</b>: gestandaardiseerde coëfficiënt <b>β</b> met richting (hypothese).</li>
          <li><b>Covariantie (↔ / ~~)</b>: samenhang zonder richting, parameter <b>ψ</b>.</li>
          <li><b>Factor loading (λ)</b>: sterkte waarmee een item het construct meet (hier: <b>Est. Std</b>).</li>
          <li><b>Residual variance</b>: deel van een item dat niet door het construct verklaard wordt
              (bij gestandaardiseerde oplossing vaak ≈ <b>1 − λ²</b>).</li>
          <li><b>Significantie</b>: * p&lt;0.05, ** p&lt;0.01, *** p&lt;0.001.</li>
        </ul>
        __FIT_BLOCK__
      </div>
    </div>
  </div>
</div>

<script>
  const BUNDLE = __BUNDLE_JSON__;
  const plotDiv = document.getElementById('plot');
  const backBtn = document.getElementById('backBtn');
  const resetBtn= document.getElementById('resetBtn');
  const searchBox = document.getElementById('searchBox');
  const crumb = document.getElementById('crumb');

  let view = "overview";
  let currentLatent = null;

  function bindOverviewEvents() {
    if (typeof plotDiv.on !== "function") return;
    if (typeof plotDiv.removeAllListeners === "function") plotDiv.removeAllListeners('plotly_click');
    plotDiv.on('plotly_click', function(evt) {
      const pt = evt && evt.points ? evt.points[0] : null;
      if (!pt) return;
      if (pt.curveNumber === BUNDLE.overviewNodeTraceIndex) {
        const key = BUNDLE.latents[pt.pointIndex];
        if (key) renderDetail(key);
      }
    });
  }

  function renderOverview() {
    view = "overview";
    currentLatent = null;
    crumb.textContent = "Overzicht";
    backBtn.style.display = "none";
    Plotly.newPlot(plotDiv, BUNDLE.overviewFig.data, BUNDLE.overviewFig.layout, {displayModeBar:true, responsive:true})
      .then(bindOverviewEvents);
  }

  function renderDetail(latentKey) {
    view = "detail";
    currentLatent = latentKey;
    crumb.textContent = "Detail: " + (BUNDLE.latentTitles[latentKey] || latentKey);
    backBtn.style.display = "inline-block";
    Plotly.newPlot(plotDiv, BUNDLE.detailFigs[latentKey].data, BUNDLE.detailFigs[latentKey].layout, {displayModeBar:true, responsive:true});
  }

  function resetZoom() {
    Plotly.relayout(plotDiv, {"xaxis.autorange": true, "yaxis.autorange": true});
  }

  function doSearch(q) {
    const query = (q || "").trim().toLowerCase();
    if (!query) return;

    if (view === "overview") {
      for (let i=0; i<BUNDLE.latents.length; i++) {
        const key = BUNDLE.latents[i];
        const title = (BUNDLE.latentTitles[key] || key).toLowerCase();
        if (title.includes(query) || key.toLowerCase().includes(query)) { renderDetail(key); return; }
      }
      return;
    }

    if (view === "detail" && currentLatent) {
      const meta = BUNDLE.detailMeta[currentLatent];
      let idx = meta.indicatorIndexByName[query];
      if (idx == null) {
        for (const k in meta.indicatorIndexByName) {
          if (k.includes(query)) { idx = meta.indicatorIndexByName[k]; break; }
        }
        if (idx == null) return;
      }
      const fig = BUNDLE.detailFigs[currentLatent];
      const x = fig.data[meta.indicatorTraceIndex].x[idx];
      const y = fig.data[meta.indicatorTraceIndex].y[idx];
      Plotly.relayout(plotDiv, {"xaxis.range": [x-5.0, x+5.0], "yaxis.range": [y-4.2, y+4.2]});
    }
  }

  backBtn.addEventListener('click', renderOverview);
  resetBtn.addEventListener('click', resetZoom);
  searchBox.addEventListener('keydown', function(e) { if (e.key === "Enter") doSearch(searchBox.value); });
  window.addEventListener("load", function() { renderOverview(); });
</script>
</body>
</html>
"""

html = HTML.replace("__PLOTLY_JS__", plotly_js)\
           .replace("__BUNDLE_JSON__", bundle_json)\
           .replace("__FIT_BLOCK__", fit_block_html)

with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
    f.write(html)

print("✅ Klaar:", OUTPUT_HTML)
print("Open door dubbelklik op sem_dashboard.html (of via file:///...)")


✅ Klaar: sem_dashboard.html
Open het bestand in je browser (dubbelklik).
